<a href="https://colab.research.google.com/github/iadicarlo/seavigil/blob/main/notebooks/sentinel1_vessel_detection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# SeaVigil: Sentinel-1 vessel detection (Allen Institute model)

Runs the open, pre-trained **Allen Institute** Sentinel-1 detector (Apache-2.0, the model behind Skylight) on a Copernicus scene, on a free Colab GPU. Output is `seavigil_detections.csv`.

**Before you run:** (1) `Runtime` -> `Change runtime type` -> **GPU**. (2) key icon (left) -> add secrets `CDSE_S3_KEY` and `CDSE_S3_SECRET` (your Copernicus S3 keys), notebook access ON. (3) `Runtime` -> `Run all`.

Note: the model is a FastAPI service with a couple of repo path quirks; the cells below fix them (config path + model paths + cached backbones) and call it the supported way (a local server we POST to).


## 1. Clone the model + pull weights (Git LFS)


In [ ]:
!sudo apt-get -qq update >/dev/null && sudo apt-get -qq install -y git-lfs >/dev/null
!git lfs install >/dev/null
import os
if not os.path.isdir('/content/vessel-detection-sentinels'):
    !git clone --depth 1 https://github.com/allenai/vessel-detection-sentinels.git /content/vessel-detection-sentinels
%cd /content/vessel-detection-sentinels
!git lfs pull --include='data/model_artifacts/sentinel-1/**' --include='torch_weights/**'
!ls -la data/model_artifacts/sentinel-1/frcnn_cmp2/3dff445/best.pth data/model_artifacts/sentinel-1/attr/c34aa37/best.pth


## 2. Dependencies
Keeps Colab's GPU PyTorch (the model loads its weights into it); installs GDAL matched to the system lib plus the model's other deps and the server bits.


In [ ]:
# Dependencies. GDAL is the one trap: build the Python binding against apt's libgdal,
# FOR COLAB'S PYTHON. (apt's python3-gdal targets the system Python; its compiled _gdal.so
# will not import under Colab's 3.12, which is the "No module named osgeo._gdal" error.)
!sudo apt-get -qq install -y libgdal-dev gdal-bin >/dev/null
!pip -q install --no-build-isolation "GDAL==$(gdal-config --version)"
# the model's other deps Colab may lack (unpinned, so they coexist with Colab's torch/numpy)
!pip -q install boto3 mapcalc lightgbm scikit-image pyproj rasterio
import torch
from osgeo import gdal
print('torch', torch.__version__, '| GPU:', torch.cuda.is_available(), '| GDAL', gdal.__version__)


## 3. Fix the repo's paths (config + weights + backbones)
The repo's config.yml points at Docker container paths and main.py looks for config one level too deep; we repoint it at the weights we just pulled and mirror it where the code expects.


In [ ]:
# One compatibility patch. The repo targets pandas 1.4, which silently upcast; pandas 2.x
# refuses to write the float length/speed/class the postprocessor produces into the detection
# DataFrame's int-seeded columns. Seeding them as float fixes it (idempotent).
import os, shutil, pathlib
ROOT = '/content/vessel-detection-sentinels'
pl = pathlib.Path(ROOT, 'src/inference/pipeline.py')
t = pl.read_text()
if 'data=[output + [0.0] * 20' not in t:
    pl.write_text(t.replace('data=[output + [0] * 20', 'data=[output + [0.0] * 20'))
    print('patched pipeline.py for pandas 2.x')
# cache the torchvision backbone where the model expects it (skips a re-download at load)
os.makedirs('/root/.cache/torch/hub/checkpoints', exist_ok=True)
for w in ['resnet50-0676ba61.pth']:
    s = f'{ROOT}/torch_weights/{w}'
    if os.path.exists(s):
        shutil.copy(s, f'/root/.cache/torch/hub/checkpoints/{w}')
print('ready')


## 4. Read an AOI straight from the scene on S3 (Cloud-Optimized, no full download)

In [ ]:
# Read just an AOI straight from the scene's Cloud-Optimized GeoTIFFs on S3 (no multi-GB
# download: gdal pulls only the COG tiles inside the bounding box). Add CDSE_S3_KEY and
# CDSE_S3_SECRET in the Colab "Secrets" panel first.
from google.colab import userdata
import boto3, math, numpy as np
from botocore.config import Config
from osgeo import gdal
gdal.DontUseExceptions()
KEY, SECRET = userdata.get('CDSE_S3_KEY'), userdata.get('CDSE_S3_SECRET')
EP = 'eodata.dataspace.copernicus.eu'
for k, v in {'AWS_S3_ENDPOINT': EP, 'AWS_VIRTUAL_HOSTING': 'FALSE', 'AWS_HTTPS': 'YES',
             'AWS_ACCESS_KEY_ID': KEY, 'AWS_SECRET_ACCESS_KEY': SECRET}.items():
    gdal.SetConfigOption(k, v)

SCENE = 'S1D_IW_GRDH_1SDV_20260627T114116_20260627T114145_003421_006075_6BE9_COG.SAFE'
W, S, E, N = -90.5, -0.85, -90.0, -0.35   # AOI lon/lat: Galapagos reserve (Santa Cruz / Baltra)

d = SCENE.split('_')[4][:8]               # YYYYMMDD from the acquisition timestamp
coll = 'IW_GRDH_1S-COG' if SCENE.endswith('_COG.SAFE') else 'IW_GRDH_1S'
prefix = f'Sentinel-1/SAR/{coll}/{d[:4]}/{d[4:6]}/{d[6:8]}/{SCENE}/measurement'
s3 = boto3.client('s3', endpoint_url=f'https://{EP}', aws_access_key_id=KEY,
                  aws_secret_access_key=SECRET, config=Config(s3={'addressing_style': 'path'}),
                  region_name='default')
objs = s3.list_objects_v2(Bucket='eodata', Prefix=prefix + '/')['Contents']
pol_key = {o['Key'].split('/')[-1].split('-')[3]: o['Key'] for o in objs
           if o['Key'].split('/')[-1].split('-')[3] in ('vh', 'vv')}
PX = 2 * math.pi * 6378137 / 512 / (2 ** 13)   # 9.5547 m (the model's preprocessing resolution)
crop = {}
for pol, key_ in pol_key.items():
    gdal.Warp(f'/content/{pol}.tif', f'/vsis3/eodata/{key_}', dstSRS='EPSG:3857', xRes=PX, yRes=PX,
              outputBounds=[W, S, E, N], outputBoundsSRS='EPSG:4326', resampleAlg='bilinear',
              multithread=True)
    crop[pol] = np.clip(gdal.Open(f'/content/{pol}.tif').ReadAsArray(), 0, 255).astype(np.uint8)
    print(pol, crop[pol].shape)


## 5. Run inference: the detector + attribute postprocessor (direct call, not the server)

In [ ]:
# Build the 6-channel input (vh, vv + 4 zero-filled overlap channels) and run the detector +
# attribute postprocessor directly on the GPU. We call the functions, not the repo's FastAPI
# server, which is broken at this commit (main.py calls detect_vessels with the wrong arg count).
import sys, os, shutil, numpy as np, torch
sys.path.insert(0, f'{ROOT}/src')
sys.path.insert(0, ROOT)
stem = SCENE.replace('.SAFE', '')
z = np.zeros_like(crop['vh'])
np.save(f'/content/{stem}_cat.npy', np.stack([crop['vh'], crop['vv'], z, z, z, z], axis=0))
shutil.copy('/content/vh.tif', f'/content/{stem}_base.tif')
meas = f'/content/raw/{stem}/measurement'
os.makedirs(meas, exist_ok=True)
shutil.copy('/content/vh.tif', f'{meas}/base.tif')

from inference.pipeline import detect_vessels
DET = f'{ROOT}/data/model_artifacts/sentinel-1/frcnn_cmp2/3dff445'
POST = f'{ROOT}/data/model_artifacts/sentinel-1/attr/c34aa37'
os.makedirs('/content/output', exist_ok=True)
dev = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
detect_vessels(DET, POST, '/content/raw', stem, f'/content/{stem}_cat.npy',
               f'/content/{stem}_base.tif', '/content/output',
               2048, 400, 20, 0.75, 10, False, dev, 'sentinel1', None)
print('inference done on', dev)


## 6. Review + download the detections
Then run `scripts/sar_detections_to_incidents.py --detections seavigil_detections.csv` in the SeaVigil repo to fold them into the map.


In [ ]:
import pandas as pd, glob
f = glob.glob('/content/output/**/predictions.csv', recursive=True)[0]
df = pd.read_csv(f)
df['scene_id'] = SCENE.replace('.SAFE', '')
print(len(df), 'vessel detections')
display(df[['lat', 'lon', 'score', 'vessel_length_m', 'is_fishing_vessel', 'vessel_speed_k']].head(25))
df.to_csv('seavigil_detections.csv', index=False)
from google.colab import files
files.download('seavigil_detections.csv')   # then: scripts/sar_detections_to_incidents.py --detections seavigil_detections.csv
